# NB_bishop_ch09_regularization

**Bishop Chapter 9 — Regularization: Weight Decay, Dropout, Early Stopping, Residual Connections & Double Descent**

This notebook explores regularization techniques for neural networks: L2 weight decay, dropout ablation, early stopping, skip/residual connections, and the double descent phenomenon.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_09/NB_bishop_ch09_regularization.ipynb)

In [ ]:
# Install dependencies (Colab-safe)
# !pip install tensorflow numpy matplotlib scikit-learn

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from sklearn.model_selection import train_test_split

print(f'TensorFlow version: {tf.__version__}')
print(f'NumPy version: {np.__version__}')

In [ ]:
# Chart styling setup
matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    """Save figure with transparent background, no grid, legend outside bottom."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

## Dataset: Noisy Polynomial Regression

We create a small dataset that is easy to overfit, then apply various regularization techniques.

In [ ]:
# Generate small noisy dataset (easy to overfit)
np.random.seed(42)
tf.random.set_seed(42)

N = 50
x_all = np.sort(np.random.uniform(-3, 3, N)).astype(np.float32).reshape(-1, 1)
y_all = (np.sin(x_all) + 0.3 * np.random.randn(N, 1)).astype(np.float32)

x_train, x_val, y_train, y_val = train_test_split(x_all, y_all, test_size=0.3, random_state=42)

# Dense test grid for plotting
x_plot = np.linspace(-3.5, 3.5, 300).astype(np.float32).reshape(-1, 1)

print(f'Train: {x_train.shape[0]} samples, Val: {x_val.shape[0]} samples')

## Part 1: Weight Decay (L2 Regularization)

Weight decay adds a penalty $\lambda \|\mathbf{w}\|^2$ to the loss function, encouraging smaller weights and smoother functions.

In [ ]:
# Train models with different weight decay strengths
lambdas = [0.0, 0.001, 0.01, 0.1]
wd_models = {}
wd_histories = {}

for lam in lambdas:
    tf.random.set_seed(42)
    reg = tf.keras.regularizers.l2(lam) if lam > 0 else None
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=reg, input_shape=(1,)),
        tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=reg),
        tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=reg),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    hist = model.fit(x_train, y_train, epochs=500, batch_size=16,
                     validation_data=(x_val, y_val), verbose=0)
    wd_models[lam] = model
    wd_histories[lam] = hist.history
    print(f'lambda={lam:.3f}: train_loss={hist.history["loss"][-1]:.4f}, val_loss={hist.history["val_loss"][-1]:.4f}')

In [ ]:
# Plot weight decay results
colors_wd = ['#cd0000', '#DAA520', '#228B22', '#1a3a6e']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Predictions
axes[0].scatter(x_train, y_train, s=30, color='gray', alpha=0.6, label='Train data', zorder=5)
axes[0].scatter(x_val, y_val, s=30, color='gray', marker='^', alpha=0.6, label='Val data', zorder=5)
axes[0].plot(x_plot, np.sin(x_plot), 'k--', lw=1, label='True function', alpha=0.5)
for lam, c in zip(lambdas, colors_wd):
    y_pred = wd_models[lam].predict(x_plot, verbose=0)
    axes[0].plot(x_plot, y_pred, color=c, lw=2, label=f'$\\lambda$={lam}')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title('Predictions with Weight Decay')
axes[0].set_ylim(-2.5, 2.5)
axes[0].legend()

# Panel 2: Validation loss
for lam, c in zip(lambdas, colors_wd):
    axes[1].plot(wd_histories[lam]['val_loss'], color=c, lw=1.5, label=f'$\\lambda$={lam}')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Val MSE')
axes[1].set_title('Validation Loss')
axes[1].set_yscale('log')
axes[1].legend()

fig.tight_layout()
save_fig(fig, 'bishop_ch9_weight_decay')
plt.show()

In [ ]:
# Analyze weight magnitudes
print(f'{"Lambda":>8s} {"Mean |W|":>12s} {"Max |W|":>12s} {"Params > 1":>12s}')
print('-' * 50)
for lam in lambdas:
    all_weights = np.concatenate([w.numpy().flatten() for w in wd_models[lam].trainable_weights
                                  if len(w.shape) == 2])
    print(f'{lam:>8.3f} {np.mean(np.abs(all_weights)):>12.4f} {np.max(np.abs(all_weights)):>12.4f} '
          f'{np.sum(np.abs(all_weights) > 1):>12d}')

## Part 2: Dropout Ablation Study

Dropout randomly sets activations to zero during training, acting as an ensemble of thinned networks.

In [ ]:
# MNIST classification for dropout experiment
(x_mnist_train, y_mnist_train), (x_mnist_test, y_mnist_test) = tf.keras.datasets.mnist.load_data()
x_mnist_train = x_mnist_train.reshape(-1, 784).astype(np.float32) / 255.0
x_mnist_test = x_mnist_test.reshape(-1, 784).astype(np.float32) / 255.0

# Use subset for speed
x_mnist_train = x_mnist_train[:5000]
y_mnist_train = y_mnist_train[:5000]

print(f'MNIST train: {x_mnist_train.shape}, test: {x_mnist_test.shape}')

In [ ]:
# Train with different dropout rates
dropout_rates = [0.0, 0.1, 0.3, 0.5, 0.7]
drop_histories = {}

for rate in dropout_rates:
    tf.random.set_seed(42)
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(256, activation='relu', input_shape=(784,)),
        tf.keras.layers.Dropout(rate),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(rate),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    hist = model.fit(x_mnist_train, y_mnist_train, epochs=50, batch_size=64,
                     validation_data=(x_mnist_test, y_mnist_test), verbose=0)
    drop_histories[rate] = hist.history
    print(f'Dropout={rate:.1f}: train_acc={hist.history["accuracy"][-1]:.4f}, '
          f'val_acc={hist.history["val_accuracy"][-1]:.4f}')

In [ ]:
# Plot dropout ablation
colors_drop = ['#cd0000', '#DAA520', '#228B22', '#1a3a6e', '#8B008B']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for rate, c in zip(dropout_rates, colors_drop):
    axes[0].plot(drop_histories[rate]['accuracy'], color=c, lw=1, alpha=0.5)
    axes[0].plot(drop_histories[rate]['val_accuracy'], color=c, lw=2, label=f'drop={rate}')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Dropout Ablation: Accuracy (solid=val, faded=train)')
axes[0].legend()

# Gap analysis
rates = []
train_accs = []
val_accs = []
gaps = []
for rate in dropout_rates:
    ta = drop_histories[rate]['accuracy'][-1]
    va = drop_histories[rate]['val_accuracy'][-1]
    rates.append(rate)
    train_accs.append(ta)
    val_accs.append(va)
    gaps.append(ta - va)

x_bar = np.arange(len(rates))
width = 0.35
axes[1].bar(x_bar - width/2, train_accs, width, label='Train', color='#1a3a6e', alpha=0.8)
axes[1].bar(x_bar + width/2, val_accs, width, label='Val', color='#cd0000', alpha=0.8)
axes[1].set_xticks(x_bar)
axes[1].set_xticklabels([f'{r:.1f}' for r in rates])
axes[1].set_xlabel('Dropout Rate')
axes[1].set_ylabel('Final Accuracy')
axes[1].set_title('Train vs Val Accuracy by Dropout Rate')
axes[1].legend()

fig.tight_layout()
save_fig(fig, 'bishop_ch9_dropout')
plt.show()

## Part 3: Early Stopping

Stop training when validation loss begins to increase.

In [ ]:
# Train with early stopping callback
tf.random.set_seed(42)
model_es = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(1,)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(1)
])
model_es.compile(optimizer='adam', loss='mse')

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=20, restore_best_weights=True, verbose=1
)

hist_es = model_es.fit(x_train, y_train, epochs=1000, batch_size=16,
                       validation_data=(x_val, y_val), callbacks=[early_stop], verbose=0)

stopped_epoch = len(hist_es.history['loss'])
best_val = min(hist_es.history['val_loss'])
print(f'Stopped at epoch {stopped_epoch}, best val_loss = {best_val:.4f}')

In [ ]:
# Also train without early stopping for comparison
tf.random.set_seed(42)
model_no_es = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(1,)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(1)
])
model_no_es.compile(optimizer='adam', loss='mse')
hist_no_es = model_no_es.fit(x_train, y_train, epochs=1000, batch_size=16,
                             validation_data=(x_val, y_val), verbose=0)

In [ ]:
# Plot early stopping
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(hist_no_es.history['loss'], color='#1a3a6e', lw=1.5, label='Train')
axes[0].plot(hist_no_es.history['val_loss'], color='#cd0000', lw=1.5, label='Val')
axes[0].axvline(stopped_epoch, color='#228B22', lw=2, ls='--', label=f'Early stop (epoch {stopped_epoch})')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_yscale('log')
axes[0].set_title('Early Stopping Criterion')
axes[0].legend()

# Predictions comparison
y_es = model_es.predict(x_plot, verbose=0)
y_no_es = model_no_es.predict(x_plot, verbose=0)

axes[1].scatter(x_train, y_train, s=30, color='gray', alpha=0.6, label='Train', zorder=5)
axes[1].scatter(x_val, y_val, s=30, color='gray', marker='^', alpha=0.6, label='Val', zorder=5)
axes[1].plot(x_plot, np.sin(x_plot), 'k--', lw=1, alpha=0.5, label='True')
axes[1].plot(x_plot, y_no_es, color='#cd0000', lw=2, label='No early stop')
axes[1].plot(x_plot, y_es, color='#228B22', lw=2, label='With early stop')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].set_title('Predictions: Early Stopping Effect')
axes[1].set_ylim(-2.5, 2.5)
axes[1].legend()

fig.tight_layout()
save_fig(fig, 'bishop_ch9_early_stopping')
plt.show()

## Part 4: Residual (Skip) Connections

Residual connections allow gradients to flow directly through the network, enabling training of very deep networks.

In [ ]:
# Build plain deep network vs ResNet-style network
def build_plain_network(n_blocks=10, units=64):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(784,)))
    model.add(tf.keras.layers.Dense(units, activation='relu'))
    for _ in range(n_blocks):
        model.add(tf.keras.layers.Dense(units, activation='relu'))
    model.add(tf.keras.layers.Dense(10, activation='softmax'))
    return model

def build_residual_network(n_blocks=10, units=64):
    inputs = tf.keras.Input(shape=(784,))
    x = tf.keras.layers.Dense(units, activation='relu')(inputs)
    for _ in range(n_blocks):
        skip = x
        x = tf.keras.layers.Dense(units, activation='relu')(x)
        x = tf.keras.layers.Dense(units)(x)
        x = tf.keras.layers.Add()([x, skip])
        x = tf.keras.layers.Activation('relu')(x)
    outputs = tf.keras.layers.Dense(10, activation='softmax')(x)
    return tf.keras.Model(inputs=inputs, outputs=outputs)

In [ ]:
# Train both architectures
res_results = {}
for name, builder in [('Plain (20 layers)', lambda: build_plain_network(20)),
                       ('Residual (20 blocks)', lambda: build_residual_network(10))]:
    tf.random.set_seed(42)
    model = builder()
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    hist = model.fit(x_mnist_train, y_mnist_train, epochs=30, batch_size=64,
                     validation_data=(x_mnist_test, y_mnist_test), verbose=0)
    res_results[name] = hist.history
    n_params = model.count_params()
    print(f'{name:>25s}: params={n_params:>7d}, val_acc={hist.history["val_accuracy"][-1]:.4f}')

In [ ]:
# Plot residual connections comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for (name, hist), c in zip(res_results.items(), ['#cd0000', '#228B22']):
    axes[0].plot(hist['loss'], color=c, lw=1, alpha=0.5)
    axes[0].plot(hist['val_loss'], color=c, lw=2, label=name)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss (solid=val, faded=train)')
axes[0].legend()

for (name, hist), c in zip(res_results.items(), ['#cd0000', '#228B22']):
    axes[1].plot(hist['val_accuracy'], color=c, lw=2, label=name)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Val Accuracy')
axes[1].set_title('Validation Accuracy')
axes[1].legend()

fig.tight_layout()
save_fig(fig, 'bishop_ch9_residual')
plt.show()

## Part 5: Double Descent

The double descent phenomenon shows that test error can decrease again as model complexity increases past the interpolation threshold.

In [ ]:
# Double descent: vary model width on a fixed small dataset
# Use a small subset to make interpolation threshold visible
np.random.seed(42)
N_dd = 100
x_dd = np.random.randn(N_dd, 10).astype(np.float32)
y_dd = (np.sum(x_dd[:, :3], axis=1) + 0.5 * np.random.randn(N_dd)).astype(np.float32)

x_dd_train, x_dd_test, y_dd_train, y_dd_test = train_test_split(
    x_dd, y_dd, test_size=0.3, random_state=42)

print(f'Double descent data: train={x_dd_train.shape[0]}, test={x_dd_test.shape[0]}, features={x_dd.shape[1]}')

In [ ]:
# Sweep model widths
widths = [2, 4, 6, 8, 10, 15, 20, 30, 50, 75, 100, 150, 200, 300, 500]
train_losses = []
test_losses = []
param_counts = []

for w in widths:
    tf.random.set_seed(42)
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(w, activation='relu', input_shape=(10,)),
        tf.keras.layers.Dense(w, activation='relu'),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    model.fit(x_dd_train, y_dd_train, epochs=2000, batch_size=32, verbose=0)
    
    tr_loss = model.evaluate(x_dd_train, y_dd_train, verbose=0)
    te_loss = model.evaluate(x_dd_test, y_dd_test, verbose=0)
    n_params = model.count_params()
    
    train_losses.append(tr_loss)
    test_losses.append(te_loss)
    param_counts.append(n_params)
    print(f'Width={w:>4d}, params={n_params:>6d}: train={tr_loss:.4f}, test={te_loss:.4f}')

In [ ]:
# Plot double descent
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(param_counts, train_losses, 'o-', color='#1a3a6e', lw=2, markersize=5, label='Train MSE')
ax.plot(param_counts, test_losses, 's-', color='#cd0000', lw=2, markersize=5, label='Test MSE')
ax.axvline(x_dd_train.shape[0], color='gray', ls='--', lw=1, label=f'n_train={x_dd_train.shape[0]}')
ax.set_xlabel('Number of Parameters')
ax.set_ylabel('MSE')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_title('Double Descent: Test Error vs Model Complexity')
ax.legend()

fig.tight_layout()
save_fig(fig, 'bishop_ch9_double_descent')
plt.show()

## Regularization Methods Comparison

In [ ]:
# Compare all regularization methods on MNIST
configs = {
    'No Regularization': {},
    'L2 (0.01)': {'kernel_regularizer': tf.keras.regularizers.l2(0.01)},
    'Dropout (0.3)': {'dropout': 0.3},
    'L2 + Dropout': {'kernel_regularizer': tf.keras.regularizers.l2(0.01), 'dropout': 0.3}
}

comp_results = {}
for name, cfg in configs.items():
    tf.random.set_seed(42)
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Dense(256, activation='relu', input_shape=(784,),
                                     kernel_regularizer=cfg.get('kernel_regularizer', None)))
    if 'dropout' in cfg:
        model.add(tf.keras.layers.Dropout(cfg['dropout']))
    model.add(tf.keras.layers.Dense(256, activation='relu',
                                     kernel_regularizer=cfg.get('kernel_regularizer', None)))
    if 'dropout' in cfg:
        model.add(tf.keras.layers.Dropout(cfg['dropout']))
    model.add(tf.keras.layers.Dense(10, activation='softmax'))
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    hist = model.fit(x_mnist_train, y_mnist_train, epochs=50, batch_size=64,
                     validation_data=(x_mnist_test, y_mnist_test), verbose=0)
    comp_results[name] = hist.history
    gap = hist.history['accuracy'][-1] - hist.history['val_accuracy'][-1]
    print(f'{name:>20s}: val_acc={hist.history["val_accuracy"][-1]:.4f}, gap={gap:.4f}')

In [ ]:
# Summary bar chart
fig, ax = plt.subplots(figsize=(9, 5))
names = list(comp_results.keys())
val_accs = [comp_results[n]['val_accuracy'][-1] for n in names]
train_accs = [comp_results[n]['accuracy'][-1] for n in names]

x_pos = np.arange(len(names))
ax.bar(x_pos - 0.2, train_accs, 0.35, label='Train Acc', color='#1a3a6e', alpha=0.8)
ax.bar(x_pos + 0.2, val_accs, 0.35, label='Val Acc', color='#cd0000', alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('Regularization Methods Comparison')
ax.legend()

fig.tight_layout()
plt.show()

## Summary

Key takeaways from Bishop Chapter 9:
- **Weight decay** (L2) penalizes large weights, producing smoother functions. Too much regularization underfits.
- **Dropout** acts as an implicit ensemble, reducing overfitting. Optimal rate is typically 0.2-0.5.
- **Early stopping** halts training when validation performance degrades — simple and effective.
- **Residual connections** enable training of very deep networks by providing gradient shortcuts.
- **Double descent** shows that test error can improve again in the overparameterized regime, challenging the classical bias-variance tradeoff.

In [ ]:
print('Notebook complete.')